# Model Walkthrough — Severe AS Detection

This notebook generates synthetic echocardiogram-like data and walks through the
two-stage model pipeline:

1. **SSL pretraining** — SimCLR + frame reordering on the 3D ResNet-18 encoder
2. **Supervised finetuning** — Binary classification (severe AS vs. not) with clip/video/study aggregation

All data is fake random tensors written as small `.avi` files so the actual dataset
and model code from the repo can be used as-is.

## 1. Setup — Generate synthetic data on disk

In [1]:
import os
import sys
import shutil
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision

# So we can import from the repo subdirectories
REPO_ROOT = os.path.abspath(".")
sys.path.insert(0, os.path.join(REPO_ROOT, "ssl_pretraining"))
sys.path.insert(0, os.path.join(REPO_ROOT, "AS_detection"))

# ---------- Configuration ----------
SYNTH_DIR = "/tmp/echo_synth"          # temp directory for synthetic data
N_STUDIES = 6                          # number of fake "patients"
VIDEOS_PER_STUDY = 3                   # PLAX clips per study
FRAME_H, FRAME_W = 112, 112           # spatial size the model expects
N_FRAMES = 32                          # frames per video (long enough to sample clips)

# ---------- Create directory structure ----------
if os.path.exists(SYNTH_DIR):
    shutil.rmtree(SYNTH_DIR)
os.makedirs(os.path.join(SYNTH_DIR, "videos"), exist_ok=True)

# ---------- Write small random .avi files ----------
rng = np.random.default_rng(2718)
rows = []

for study_idx in range(N_STUDIES):
    acc_num = f"STUDY_{study_idx:04d}"
    # ~half severe, half not
    is_severe = int(study_idx < N_STUDIES // 2)
    av_stenosis = "Severe" if is_severe else "None"

    for vid_idx in range(VIDEOS_PER_STUDY):
        fname = f"{acc_num}_vid{vid_idx}.avi"
        fpath = os.path.join(SYNTH_DIR, "videos", fname)

        fourcc = cv2.VideoWriter_fourcc(*"MJPG")
        writer = cv2.VideoWriter(fpath, fourcc, 30, (FRAME_W, FRAME_H))
        for _ in range(N_FRAMES):
            frame = rng.integers(0, 256, size=(FRAME_H, FRAME_W, 3), dtype=np.uint8)
            writer.write(frame)
        writer.release()

        rows.append({
            "plax_prob": round(rng.uniform(0.8, 1.0), 3),
            "fpath": fname,
            "acc_num": acc_num,
            "av_stenosis": av_stenosis,
            "video_num": vid_idx,
            "severe_AS": is_severe,
        })

# ---------- Write split CSVs (same data for train/val/test) ----------
df = pd.DataFrame(rows)
for split in [
    "100122_train_2016-2020",
    "100122_val_2016-2020",
    "051823_full_test_2016-2020",
    "100122_test_2021",
]:
    df.to_csv(os.path.join(SYNTH_DIR, f"{split}.csv"), index=False)

print(f"Created {len(rows)} synthetic videos in {SYNTH_DIR}")
df

Created 18 synthetic videos in /tmp/echo_synth


,plax_prob,fpath,acc_num,av_stenosis,video_num,severe_AS
0,0.918,STUDY_0000_vid0.avi,STUDY_0000,Severe,0,1
1,0.862,STUDY_0000_vid1.avi,STUDY_0000,Severe,1,1
2,0.980,STUDY_0000_vid2.avi,STUDY_0000,Severe,2,1
3,0.892,STUDY_0001_vid0.avi,STUDY_0001,Severe,0,1
4,0.920,STUDY_0001_vid1.avi,STUDY_0001,Severe,1,1
5,0.980,STUDY_0001_vid2.avi,STUDY_0001,Severe,2,1
6,0.857,STUDY_0002_vid0.avi,STUDY_0002,Severe,0,1
7,0.894,STUDY_0002_vid1.avi,STUDY_0002,Severe,1,1
8,0.855,STUDY_0002_vid2.avi,STUDY_0002,Severe,2,1
9,0.908,STUDY_0003_vid0.avi,STUDY_0003,None,0,0


## 2. Stage 1 — SSL Pretraining

The SSL dataset pairs two *different* videos from the *same* study as a positive
pair. Each video is clipped to 4 frames, augmented, and its frames are randomly
permuted. The model has to (a) pull the two views together via contrastive loss
and (b) predict which of the 4! = 24 permutations was applied (frame reordering).

In [2]:
from ssl_pretraining.dataset import EchoDataset as SSLDataset
from ssl_pretraining.model import SimCLR
from ssl_pretraining.losses import NT_Xent

# --- Dataset ---
ssl_ds = SSLDataset(
    data_dir=SYNTH_DIR,
    split="100122_train_2016-2020",
    clip_len=4,       # 4 frames per clip → 4! = 24 possible orderings
    sampling_rate=1,
)

print(f"SSL dataset size: {len(ssl_ds)} video pairs")
print(f"Number of frame orderings (classes): {len(ssl_ds.temporal_orderings)}")

# Grab one sample
x_i, x_j, t_i, t_j = ssl_ds[0]
print(f"\nSample shapes:")
print(f"  x_i (clip 1): {x_i.shape}  — (C, T, H, W)")
print(f"  x_j (clip 2): {x_j.shape}")
print(f"  t_i (ordering label 1): {t_i}  — index into the 24 permutations")
print(f"  t_j (ordering label 2): {t_j}")

100%|██████████| 6/6 [00:00<00:00, 958.00it/s]

SSL dataset size: 18 video pairs
Number of frame orderings (classes): 24

Sample shapes:
  x_i (clip 1): torch.Size([3, 4, 112, 112])  — (C, T, H, W)
  x_j (clip 2): torch.Size([3, 4, 112, 112])
  t_i (ordering label 1): 12  — index into the 24 permutations
  t_j (ordering label 2): 13


In [3]:
# --- Model ---
encoder = torchvision.models.video.r3d_18(pretrained=False)
n_features = encoder.fc.in_features   # 512
print(f"Encoder feature dim: {n_features}")

ssl_model = SimCLR(
    encoder=encoder,
    projection_dim=128,
    n_features=n_features,
)

# --- Forward pass (single pair, no GPU needed) ---
# Add batch dimension
x_i_batch = x_i.unsqueeze(0)  # (1, 3, 4, 112, 112)
x_j_batch = x_j.unsqueeze(0)

with torch.no_grad():
    h_i, h_j, z_i, z_j, t_hat_i, t_hat_j = ssl_model(x_i_batch, x_j_batch)

print(f"\nh_i (encoder features):      {h_i.shape}  — 512-d representation")
print(f"z_i (projected embedding):   {z_i.shape}  — used for contrastive loss")
print(f"t_hat_i (reordering logits): {t_hat_i.shape}  — 24-class prediction")

/Users/howardbaek/Dropbox/Mac/Documents/echo-severe-as/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/howardbaek/Dropbox/Mac/Documents/echo-severe-as/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Encoder feature dim: 512

h_i (encoder features):      torch.Size([1, 512])  — 512-d representation
z_i (projected embedding):   torch.Size([1, 128])  — used for contrastive loss
t_hat_i (reordering logits): torch.Size([1, 24])  — 24-class prediction


In [4]:
# --- SSL Loss ---
# Build a small batch (batch_size=2) to demonstrate the loss
ssl_loader = torch.utils.data.DataLoader(ssl_ds, batch_size=2, shuffle=True)
x_i_b, x_j_b, t_i_b, t_j_b = next(iter(ssl_loader))

with torch.no_grad():
    h_i, h_j, z_i, z_j, t_hat_i, t_hat_j = ssl_model(x_i_b, x_j_b)

# NT-Xent contrastive loss on the projected embeddings
contrastive_loss_fn = NT_Xent(batch_size=2, temperature=0.05)
contrastive_loss = contrastive_loss_fn(z_i, z_j)

# Cross-entropy on frame reordering predictions
reorder_loss_fn = torch.nn.CrossEntropyLoss()
reorder_loss = reorder_loss_fn(
    torch.cat([t_hat_i, t_hat_j]),
    torch.cat([t_i_b, t_j_b]),
)

total_loss = contrastive_loss + reorder_loss

print(f"NT-Xent (contrastive) loss: {contrastive_loss.item():.4f}")
print(f"Frame reordering CE loss:   {reorder_loss.item():.4f}")
print(f"Total SSL loss:             {total_loss.item():.4f}")

NT-Xent (contrastive) loss: 1.0497
Frame reordering CE loss:   3.3327
Total SSL loss:             4.3823


## 3. Stage 2 — Supervised Finetuning

After SSL pretraining, only the **encoder** weights are kept. The projection head
and reordering head are discarded. A new single-logit classification head
(`Linear(512, 1)` + Dropout) is attached for binary severe-AS detection.

At training time, one random 16-frame clip is sampled per video.
At inference, 4 evenly-spaced clips are sampled and their sigmoid outputs are averaged.

In [5]:
# Use the AS_detection dataset (different from the SSL one)
from AS_detection.dataset import EchoDataset as ASDataset

# --- Training dataset (one random clip per video) ---
train_ds = ASDataset(
    data_dir=SYNTH_DIR,
    split="100122_train_2016-2020",
    clip_len=16,
    sampling_rate=1,
    num_clips=4,
    augment=True,
    kinetics=False,  # no Kinetics normalization when using SSL init
)

sample = train_ds[0]
print("Training sample:")
print(f"  x shape: {sample['x'].shape}  — (C, T, H, W), single clip")
print(f"  y:       {sample['y']}  — label (1=severe)")
print(f"  acc_num: {sample['acc_num']}")

# --- Validation dataset (4 clips per video) ---
val_ds = ASDataset(
    data_dir=SYNTH_DIR,
    split="100122_val_2016-2020",
    clip_len=16,
    sampling_rate=1,
    num_clips=4,
    kinetics=False,
)

sample_val = val_ds[0]
print(f"\nValidation sample:")
print(f"  x shape: {sample_val['x'].shape}  — (C, num_clips, T, H, W)")
print(f"  The extra dim is {sample_val['x'].shape[1]} clips to be scored independently and averaged")

severe_AS
1    9
0    9
Name: count, dtype: int64
Training sample:
  x shape: torch.Size([3, 16, 112, 112])  — (C, T, H, W), single clip
  y:       tensor([1.])  — label (1=severe)
  acc_num: STUDY_0000
severe_AS
1    9
0    9
Name: count, dtype: int64

Validation sample:
  x shape: torch.Size([3, 4, 16, 112, 112])  — (C, num_clips, T, H, W)
  The extra dim is 4 clips to be scored independently and averaged


In [6]:
# --- Build the supervised model ---
# In the real pipeline, encoder weights come from the SSL checkpoint.
# Here we just show the architecture with random init.
sup_model = torchvision.models.video.r3d_18(weights=None)
sup_model.fc = torch.nn.Sequential(
    torch.nn.Linear(512, 1),
    torch.nn.Dropout(0.25),
)

# --- Training forward pass (single clip) ---
x_train = sample["x"].unsqueeze(0)        # (1, 3, 16, 112, 112)
y_train = sample["y"].unsqueeze(0)         # (1, 1)

with torch.no_grad():
    logit = sup_model(x_train)             # raw logit
    prob = logit.sigmoid()                 # probability of severe AS

print("Training forward pass (1 clip):")
print(f"  Input:  {x_train.shape}")
print(f"  Logit:  {logit.item():.4f}")
print(f"  P(severe AS): {prob.item():.4f}")

# --- BCE loss with class weighting ---
loss_fn = torch.nn.BCEWithLogitsLoss()
loss = loss_fn(logit, y_train)
print(f"  BCE loss: {loss.item():.4f}")

Training forward pass (1 clip):
  Input:  torch.Size([1, 3, 16, 112, 112])
  Logit:  0.2153
  P(severe AS): 0.5536
  BCE loss: 0.5913


## 4. Inference Aggregation: Clips → Video → Study

At test time predictions are averaged at two levels:
1. **Clip → video**: sigmoid each of the 4 clips, average them
2. **Video → study**: average all video-level probabilities sharing the same `acc_num`

This is how the paper's study-level AUROC is computed.

In [7]:
# --- Inference on the validation set ---
val_loader = torch.utils.data.DataLoader(val_ds, batch_size=4, shuffle=False)
sup_model.eval()

all_preds, all_labels, all_acc_nums = [], [], []

with torch.no_grad():
    for batch in val_loader:
        x = batch["x"]            # (B, C, num_clips, T, H, W)
        y = batch["y"]
        acc_num = batch["acc_num"]

        # Score each clip independently, then average — this matches
        # the logic in AS_detection/utils.py evaluate()
        clip_logits = torch.stack(
            [sup_model(x[:, :, clip, :, :, :]) for clip in range(x.shape[2])],
            dim=0,
        )                          # (num_clips, B, 1)
        video_prob = clip_logits.sigmoid().mean(dim=0)  # (B, 1)

        all_preds.append(video_prob.numpy().ravel())
        all_labels.append(y.numpy().ravel())
        all_acc_nums.extend(acc_num)

# --- Video-level predictions ---
video_df = pd.DataFrame({
    "acc_num": all_acc_nums,
    "y_true": np.concatenate(all_labels),
    "y_hat": np.concatenate(all_preds),
})
print("Video-level predictions:")
print(video_df.to_string(index=False))

# --- Study-level aggregation ---
study_df = video_df.groupby("acc_num").agg({"y_true": "mean", "y_hat": "mean"})
print("\nStudy-level predictions (video probs averaged per patient):")
print(study_df.to_string())

Video-level predictions:
   acc_num  y_true    y_hat
STUDY_0000     1.0 0.505799
STUDY_0000     1.0 0.505771
STUDY_0000     1.0 0.505992
STUDY_0001     1.0 0.506382
STUDY_0001     1.0 0.505758
STUDY_0001     1.0 0.505910
STUDY_0002     1.0 0.506278
STUDY_0002     1.0 0.505704
STUDY_0002     1.0 0.506059
STUDY_0003     0.0 0.505885
STUDY_0003     0.0 0.506062
STUDY_0003     0.0 0.505966
STUDY_0004     0.0 0.506452
STUDY_0004     0.0 0.505756
STUDY_0004     0.0 0.506066
STUDY_0005     0.0 0.506316
STUDY_0005     0.0 0.505975
STUDY_0005     0.0 0.506356

Study-level predictions (video probs averaged per patient):
            y_true     y_hat
acc_num                     
STUDY_0000     1.0  0.505854
STUDY_0001     1.0  0.506017
STUDY_0002     1.0  0.506014
STUDY_0003     0.0  0.505971
STUDY_0004     0.0  0.506092
STUDY_0005     0.0  0.506216


## 5. Architecture Summary

Quick reference for the full model topology at each stage.

In [8]:
# SSL model architecture
print("=== SSL Model (SimCLR wrapper) ===\n")
print(ssl_model)
print("\n\n=== Supervised Model (finetuned for AS detection) ===\n")
print(sup_model)

=== SSL Model (SimCLR wrapper) ===

SimCLR(
  (encoder): VideoResNet(
    (stem): BasicStem(
      (0): Conv3d(3, 64, kernel_size=(3, 7, 7), stride=(1, 2, 2), padding=(1, 3, 3), bias=False)
      (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Sequential(
          (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
          (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (conv2): Sequential(
          (0): Conv3DSimple(64, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
          (1): BatchNorm3d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
        )
        (relu): ReLU(inplace=True)
      )
      (1): BasicBlock(
        (conv1): 

## 6. Deep Dive: Frame Reordering Pretext Task

The frame reordering head is the key innovation. Let's trace through exactly what happens when you feed a shuffled clip through it.

In [9]:
import itertools

# --- Step 1: All possible permutations of 4 frames ---
# The dataset has clip_len=4, so there are 4! = 24 possible orderings
clip_len = 4
all_orderings = list(itertools.permutations(range(clip_len)))

print(f"Total permutations of {clip_len} frames: {len(all_orderings)}")
print("\nFirst 5 permutations:")
for i, perm in enumerate(all_orderings[:5]):
    print(f"  Ordering {i}: {perm}")
print("  ...")
print(f"  Ordering 23: {all_orderings[23]}")

# Example: original frames in sequence [A, B, C, D]
# If we pick ordering #5, we get: [B, A, D, C]
example_idx = 5
example_perm = all_orderings[example_idx]
print(f"\n--- Example: Pick ordering {example_idx} ---")
print(f"Original frames: [A, B, C, D]  (indices: [0, 1, 2, 3])")
print(f"Permutation {example_idx}: {example_perm}")
print(f"Shuffled frames: {[chr(65 + i) for i in example_perm]}  (rearranged according to permutation)")
print(f"→ The model sees [B, A, D, C] but needs to predict '{example_idx}' to get back to [A, B, C, D]")

Total permutations of 4 frames: 24

First 5 permutations:
  Ordering 0: (0, 1, 2, 3)
  Ordering 1: (0, 1, 3, 2)
  Ordering 2: (0, 2, 1, 3)
  Ordering 3: (0, 2, 3, 1)
  Ordering 4: (0, 3, 1, 2)
  ...
  Ordering 23: (3, 2, 1, 0)

--- Example: Pick ordering 5 ---
Original frames: [A, B, C, D]  (indices: [0, 1, 2, 3])
Permutation 5: (0, 3, 2, 1)
Shuffled frames: ['A', 'D', 'C', 'B']  (rearranged according to permutation)
→ The model sees [B, A, D, C] but needs to predict '5' to get back to [A, B, C, D]


In [10]:
# --- Step 2: What the dataset returns ---
print("=== DATASET OUTPUT ===\n")

# Grab a real sample from our synthetic dataset
x_i, x_j, t_i, t_j = ssl_ds[0]

print(f"Batch item from dataset:")
print(f"  x_i shape: {x_i.shape}  ← shuffled clip 1 (C, T, H, W)")
print(f"  x_j shape: {x_j.shape}  ← shuffled clip 2")
print(f"  t_i: {t_i.item()}  ← label = which of the 24 permutations was applied to clip 1")
print(f"  t_j: {t_j.item()}  ← label = which of the 24 permutations was applied to clip 2")

print(f"\nInterpretation:")
print(f"  x_i is shuffled according to ordering #{t_i.item()}")
print(f"    Permutation: {all_orderings[t_i.item()]}")
print(f"  x_j is shuffled according to ordering #{t_j.item()}")
print(f"    Permutation: {all_orderings[t_j.item()]}")

=== DATASET OUTPUT ===

Batch item from dataset:
  x_i shape: torch.Size([3, 4, 112, 112])  ← shuffled clip 1 (C, T, H, W)
  x_j shape: torch.Size([3, 4, 112, 112])  ← shuffled clip 2
  t_i: 22  ← label = which of the 24 permutations was applied to clip 1
  t_j: 7  ← label = which of the 24 permutations was applied to clip 2

Interpretation:
  x_i is shuffled according to ordering #22
    Permutation: (3, 2, 0, 1)
  x_j is shuffled according to ordering #7
    Permutation: (1, 0, 3, 2)


In [11]:
# --- Step 3: Forward pass through the reordering head ---
print("=== MODEL FORWARD PASS ===\n")

# Create a batch
x_i_batch = x_i.unsqueeze(0)
x_j_batch = x_j.unsqueeze(0)
t_i_batch = t_i.unsqueeze(0)
t_j_batch = t_j.unsqueeze(0)

print(f"Input batch:")
print(f"  x_i_batch: {x_i_batch.shape}  ← (B=1, C, T, H, W)")
print(f"  t_i_batch (ground truth): {t_i_batch.item()}")

# Forward through encoder
with torch.no_grad():
    h_i = ssl_model.encoder(x_i_batch)
    print(f"\nAfter encoder:")
    print(f"  h_i shape: {h_i.shape}  ← (B=1, 512) feature vector")

# Forward through reordering head
with torch.no_grad():
    t_hat_i = ssl_model.reordering_head(h_i)
    print(f"\nAfter reordering head:")
    print(f"  t_hat_i shape: {t_hat_i.shape}  ← (B=1, 24) logits for each permutation")
    print(f"  t_hat_i values: {t_hat_i[0].numpy()}")

# Softmax to get probabilities
with torch.no_grad():
    probs = torch.softmax(t_hat_i, dim=1)
    predicted_label = torch.argmax(probs, dim=1)
    confidence = probs[0, predicted_label[0]]
    
print(f"\nSoftmax probabilities:")
print(f"  Predicted label: {predicted_label.item()}  (argmax of logits)")
print(f"  Ground truth label: {t_i_batch.item()}")
print(f"  Predicted correctly: {predicted_label.item() == t_i_batch.item()}")
print(f"  Confidence: {confidence.item():.4f}")
print(f"  Top 3 predictions:")
top3_probs, top3_indices = torch.topk(probs[0], 3)
for rank, (idx, prob) in enumerate(zip(top3_indices, top3_probs), 1):
    match = "✓ CORRECT" if idx == t_i_batch.item() else ""
    print(f"    {rank}. Ordering {idx.item()}: {prob.item():.4f} {match}")

=== MODEL FORWARD PASS ===

Input batch:
  x_i_batch: torch.Size([1, 3, 4, 112, 112])  ← (B=1, C, T, H, W)
  t_i_batch (ground truth): 22

After encoder:
  h_i shape: torch.Size([1, 512])  ← (B=1, 512) feature vector

After reordering head:
  t_hat_i shape: torch.Size([1, 24])  ← (B=1, 24) logits for each permutation
  t_hat_i values: [-0.40942395  0.6564483   0.11681725  0.33951008  0.1817038  -0.3894134   0.1470606
  0.9622592  -0.36509112 -0.14451538 -0.00931094  0.49418032  0.23296268 -0.19612747
  0.14749658 -0.37378547 -0.26257673  0.08351594 -0.19130154 -0.9572145   0.2880682
  0.05504994 -0.35437     0.4094155 ]

Softmax probabilities:
  Predicted label: 7  (argmax of logits)
  Ground truth label: 22
  Predicted correctly: False
  Confidence: 0.0986
  Top 3 predictions:
    1. Ordering 7: 0.0986 
    2. Ordering 1: 0.0726 
    3. Ordering 11: 0.0617 


In [12]:
# --- Step 4: The cross-entropy loss ---
print("=== CROSS-ENTROPY LOSS ===\n")

ce_loss_fn = torch.nn.CrossEntropyLoss()

# Compute loss for a single clip
loss_i = ce_loss_fn(t_hat_i, t_i_batch)

print(f"t_hat_i (logits):  {t_hat_i.shape}  ← unnormalized scores for 24 classes")
print(f"t_i_batch (label): {t_i_batch.shape}, value={t_i_batch.item()} ← ground truth class")
print(f"\nCrossEntropy({t_hat_i.shape}, {t_i_batch.shape}):")
print(f"  Loss: {loss_i.item():.4f}")

print(f"\nWhat this loss does:")
print(f"  1. Converts logits to softmax probabilities")
print(f"  2. Takes log of the probability for the ground-truth class (22)")
print(f"  3. Negates and averages (lower is better)")
print(f"\nInterpretation:")
print(f"  • Predicted class: 7 with prob 0.0986")
print(f"  • Ground truth: class 22")
print(f"  • The model is WRONG, so CE loss is HIGH")
print(f"  • During training, gradients flow back to make class 22 more likely")

# Now compute for a batch
ssl_loader = torch.utils.data.DataLoader(ssl_ds, batch_size=4, shuffle=True)
x_i_b, x_j_b, t_i_b, t_j_b = next(iter(ssl_loader))

with torch.no_grad():
    h_i, h_j, z_i, z_j, t_hat_i, t_hat_j = ssl_model(x_i_b, x_j_b)

# Concatenate both clips and both labels (like the training loop does)
all_logits = torch.cat([t_hat_i, t_hat_j])  # (8, 24)
all_labels = torch.cat([t_i_b, t_j_b])      # (8,)

batch_loss = ce_loss_fn(all_logits, all_labels)

print(f"\n--- In a batch ---")
print(f"Batch size: 4 pairs = 2 × 4 = 8 clips")
print(f"  all_logits: {all_logits.shape}  ← logits for all 8 clips")
print(f"  all_labels: {all_labels.shape}  ← ground truth for all 8 clips")
print(f"  Batch CE loss: {batch_loss.item():.4f}")
print(f"\nDuring training:")
print(f"  Total loss = NT-Xent(z_i, z_j) + CrossEntropy(logits, labels)")
print(f"  Both losses guide the encoder to learn good spatiotemporal features")

=== CROSS-ENTROPY LOSS ===

t_hat_i (logits):  torch.Size([1, 24])  ← unnormalized scores for 24 classes
t_i_batch (label): torch.Size([1]), value=22 ← ground truth class

CrossEntropy(torch.Size([1, 24]), torch.Size([1])):
  Loss: 3.6337

What this loss does:
  1. Converts logits to softmax probabilities
  2. Takes log of the probability for the ground-truth class (22)
  3. Negates and averages (lower is better)

Interpretation:
  • Predicted class: 7 with prob 0.0986
  • Ground truth: class 22
  • The model is WRONG, so CE loss is HIGH
  • During training, gradients flow back to make class 22 more likely

--- In a batch ---
Batch size: 4 pairs = 2 × 4 = 8 clips
  all_logits: torch.Size([8, 24])  ← logits for all 8 clips
  all_labels: torch.Size([8])  ← ground truth for all 8 clips
  Batch CE loss: 3.1447

During training:
  Total loss = NT-Xent(z_i, z_j) + CrossEntropy(logits, labels)
  Both losses guide the encoder to learn good spatiotemporal features


## 7. Understanding Frame Permutations

Let's see exactly what kind of reordering happens to the frames.

In [13]:
# --- How permutations are generated ---
print("=== GENERATING PERMUTATIONS ===\n")

import itertools
import numpy as np

clip_len = 4

# Line 79 in dataset.py:
# self.temporal_orderings = [_ for _ in itertools.permutations(np.arange(self.clip_len))]

temporal_orderings = list(itertools.permutations(np.arange(clip_len)))

print(f"itertools.permutations(np.arange({clip_len}))")
print(f"  = itertools.permutations([0, 1, 2, 3])")
print(f"\nGenerates {len(temporal_orderings)} permutations (all {clip_len}! orderings):\n")

for i, perm in enumerate(temporal_orderings):
    print(f"  {i:2d}: {perm}")


=== GENERATING PERMUTATIONS ===

itertools.permutations(np.arange(4))
  = itertools.permutations([0, 1, 2, 3])

Generates 24 permutations (all 4! orderings):

   0: (np.int64(0), np.int64(1), np.int64(2), np.int64(3))
   1: (np.int64(0), np.int64(1), np.int64(3), np.int64(2))
   2: (np.int64(0), np.int64(2), np.int64(1), np.int64(3))
   3: (np.int64(0), np.int64(2), np.int64(3), np.int64(1))
   4: (np.int64(0), np.int64(3), np.int64(1), np.int64(2))
   5: (np.int64(0), np.int64(3), np.int64(2), np.int64(1))
   6: (np.int64(1), np.int64(0), np.int64(2), np.int64(3))
   7: (np.int64(1), np.int64(0), np.int64(3), np.int64(2))
   8: (np.int64(1), np.int64(2), np.int64(0), np.int64(3))
   9: (np.int64(1), np.int64(2), np.int64(3), np.int64(0))
  10: (np.int64(1), np.int64(3), np.int64(0), np.int64(2))
  11: (np.int64(1), np.int64(3), np.int64(2), np.int64(0))
  12: (np.int64(2), np.int64(0), np.int64(1), np.int64(3))
  13: (np.int64(2), np.int64(0), np.int64(3), np.int64(1))
  14: (np.int64

In [14]:
# --- How the permutation is applied (from dataset.py lines 111-114) ---
print("=== APPLYING A PERMUTATION ===\n")

# Suppose we have a 4-frame clip
clip_frames = np.array(['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3'], dtype=object)
print(f"Original clip (4 frames in temporal order):")
print(f"  {clip_frames}\n")

# Randomly pick one of the 24 permutations
reordering_label = 7
reordering = temporal_orderings[reordering_label]

print(f"Pick permutation #{reordering_label}: {reordering}")
print(f"  This tuple says: 'put frame at index 1, then frame at index 0, then frame at index 3, then frame at index 2'\n")

# Apply the permutation (line 114: x = x[reordering, :, :, :])
shuffled_frames = clip_frames[list(reordering)]

print(f"Shuffled clip:")
print(f"  {shuffled_frames}\n")

print(f"Interpretation:")
print(f"  Original order:  [Frame_0, Frame_1, Frame_2, Frame_3]  (temporal sequence)")
print(f"  After shuffling: [Frame_1, Frame_0, Frame_3, Frame_2]  (out of order)")
print(f"  The model gets the shuffled clip and must predict 'permutation #7' to understand it was shuffled")

=== APPLYING A PERMUTATION ===

Original clip (4 frames in temporal order):
  ['Frame_0' 'Frame_1' 'Frame_2' 'Frame_3']

Pick permutation #7: (np.int64(1), np.int64(0), np.int64(3), np.int64(2))
  This tuple says: 'put frame at index 1, then frame at index 0, then frame at index 3, then frame at index 2'

Shuffled clip:
  ['Frame_1' 'Frame_0' 'Frame_3' 'Frame_2']

Interpretation:
  Original order:  [Frame_0, Frame_1, Frame_2, Frame_3]  (temporal sequence)
  After shuffling: [Frame_1, Frame_0, Frame_3, Frame_2]  (out of order)
  The model gets the shuffled clip and must predict 'permutation #7' to understand it was shuffled


In [15]:
# --- Concrete visual example with different permutations ---
print("=== EXAMPLES: SAME CLIP, DIFFERENT PERMUTATIONS ===\n")

examples = [0, 7, 23]

for example_idx in examples:
    perm = temporal_orderings[example_idx]
    shuffled = clip_frames[list(perm)]
    
    print(f"Permutation #{example_idx}: {perm}")
    print(f"  Original:  ['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3']")
    print(f"  Shuffled:  {list(shuffled)}")
    
    # Explain what happened
    changes = []
    for new_pos, old_pos in enumerate(perm):
        if old_pos != new_pos:
            changes.append(f"Frame_{old_pos}→pos{new_pos}")
    
    if not changes:
        print(f"  Explanation: No change (identity permutation)")
    else:
        print(f"  Explanation: {', '.join(changes)}")
    print()

=== EXAMPLES: SAME CLIP, DIFFERENT PERMUTATIONS ===

Permutation #0: (np.int64(0), np.int64(1), np.int64(2), np.int64(3))
  Original:  ['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3']
  Shuffled:  ['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3']
  Explanation: No change (identity permutation)

Permutation #7: (np.int64(1), np.int64(0), np.int64(3), np.int64(2))
  Original:  ['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3']
  Shuffled:  ['Frame_1', 'Frame_0', 'Frame_3', 'Frame_2']
  Explanation: Frame_1→pos0, Frame_0→pos1, Frame_3→pos2, Frame_2→pos3

Permutation #23: (np.int64(3), np.int64(2), np.int64(1), np.int64(0))
  Original:  ['Frame_0', 'Frame_1', 'Frame_2', 'Frame_3']
  Shuffled:  ['Frame_3', 'Frame_2', 'Frame_1', 'Frame_0']
  Explanation: Frame_3→pos0, Frame_2→pos1, Frame_1→pos2, Frame_0→pos3



In [ ]:
# --- Cleanup ---
shutil.rmtree(SYNTH_DIR)
print(f"Cleaned up {SYNTH_DIR}")